In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 6.8 MB/s eta 0:00:00


In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from scipy.linalg import toeplitz
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GroupKFold
RANDOM_STATE = 42


I had to remake the dataframe and the features I made because it was leaking data as SVD uses information from the entire matrix. My solution was to separate by block, and by patient 10s, do GMS then restack these features.

In [ ]:
def make_H(N, tau1, tau2):
    t = np.arange(N)
    h = np.exp(-t / tau1) - np.exp(-t / tau2)
    H = toeplitz(h)
    return np.tril(H)


def soft_threshold(X, tau):
    return np.sign(X) * np.maximum(np.abs(X) - tau, 0)


def svt(X, tau):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    s_thresh = np.maximum(s - tau, 0)
    return U @ np.diag(s_thresh) @ Vt


def gen_matrix_sep(M, H, lam, max_iter=100, tol=1e-6, rho=1.0):
    N, K = M.shape
    S = np.zeros((H.shape[1], K))
    L = np.zeros_like(M)
    Y_dual = np.zeros_like(M)
    HtH = H.T @ H
    inv_mat = np.linalg.inv(HtH + np.eye(H.shape[1]))
    for it in range(max_iter):
        L = svt(M - H @ S + (1 / rho) * Y_dual, 1 / rho)
        temp = H.T @ (M - L + (1 / rho) * Y_dual)
        S = inv_mat @ soft_threshold(temp, lam / rho)
        residual = M - L - H @ S
        Y_dual = Y_dual + rho * residual
        if np.linalg.norm(residual, "fro") / np.linalg.norm(M, "fro") < tol:
            break
    return {"L": L, "S": S}


def gen_matrix_sep_con(Y, H, lam):
    U, s, Vt = np.linalg.svd(H, full_matrices=False)
    rank = np.sum(s > 1e-8)
    Uk = U[:, :rank]
    Vk = Vt[:rank, :].T
    s_inv = 1 / s[:rank]
    C = Uk @ np.diag(s_inv) @ Uk.T
    CH = Uk @ Vk.T
    out = gen_matrix_sep(C @ Y, CH, lam)
    S = out["S"]
    L = Y - H @ S
    return {"S": S, "L": L}

This is where I pulled out the raw signals to do above and filled the NAs at the end of each row with median values to work with lightgbm

In [ ]:
df_raw = pd.read_csv("Raw_Features.csv")
signal_cols = [f"V{i}" for i in range(1, 41)]
#I had to create a subject ID directly to keep track of patients for the new loops
subject_id_raw = df_raw["X"].astype(str).apply(lambda x: x.split("_")[2])

row_means = df_raw[signal_cols].mean(axis=1)
X_raw_filled_full = df_raw[signal_cols].apply(
    lambda row: row.fillna(row_means[row.name]), axis=1
)

In [ ]:
df_blocks = df_raw.copy()
df_blocks["subject_id"] = subject_id_raw

df_blocks["block_change"] = (
    df_blocks.groupby("subject_id")["condition"]
    .apply(lambda s: s != s.shift())
    .reset_index(level=0, drop=True)
)
df_blocks["block_id"] = df_blocks.groupby("subject_id")["block_change"].cumsum()

tau1, tau2 = 10, 0.5
H40 = make_H(40, tau1, tau2)

phasic_rows, tonic_rows, row_order_index = [], [], []
small_block_warnings = []

for (subj, block), block_df in df_blocks.groupby(["subject_id", "block_id"]):
    rows = X_raw_filled_full.loc[block_df.index]
    Y_block = rows.values.T
    K_block = Y_block.shape[1]

    if K_block < 3:
        small_block_warnings.append((subj, block, K_block))

    lam = 3 / np.sqrt(max(40, K_block))
    out = gen_matrix_sep_con(Y_block, H40, lam)

    S_block, L_block = out["S"], out["L"]
    S_block[S_block < 0] = 0
    L_block[L_block < 0] = 0

    phasic_rows.append(S_block.T)
    tonic_rows.append(L_block.T)
    row_order_index.extend(rows.index.tolist())

if small_block_warnings:
    print(f"WARNING: {len(small_block_warnings)} blocks have <3 windows:")
    print(small_block_warnings[:10])

phasic_matrix = np.vstack(phasic_rows)
tonic_matrix = np.vstack(tonic_rows)

phasic_df = pd.DataFrame(
    phasic_matrix, index=row_order_index, columns=[f"Phasic_{c}" for c in signal_cols]
).loc[df_raw.index]
tonic_df = pd.DataFrame(
    tonic_matrix, index=row_order_index, columns=[f"Tonic_{c}" for c in signal_cols]
).loc[df_raw.index]

descriptives = pd.DataFrame(
    {
        "Mean_Phasic": phasic_df.mean(axis=1),
        "Max_Phasic": phasic_df.max(axis=1),
        "Min_Phasic": phasic_df.min(axis=1),
        "SD_Phasic": phasic_df.std(axis=1),
        "Mean_Tonic": tonic_df.mean(axis=1),
        "Max_Tonic": tonic_df.max(axis=1),
        "Min_Tonic": tonic_df.min(axis=1),
        "SD_Tonic": tonic_df.std(axis=1),
    }
)

keep_cols = ["X", "condition"] + [
    c
    for c in df_raw.columns
    if c not in signal_cols + list(descriptives.columns) + ["X", "condition"]
]
df = pd.concat([df_raw[keep_cols], descriptives], axis=1)
df = df.merge(df_raw[["X"] + signal_cols], on="X", how="left")
assert len(df) == len(df_raw), "Row count changed unexpectedly during GMS rebuild"

df.to_csv("Chen_Final_Result_PerSubjectPerBlock_GMS.csv", index=False)

This was to verify the patients didn't share informatio  anymore. Before I split up the GMS the correlation was 1

In [ ]:
subject_id = df["X"].astype(str).apply(lambda x: x.split("_")[2])
time_idx_check = df.groupby(subject_id).cumcount()
pivot_check = df.assign(t=time_idx_check).pivot_table(
    index="t", columns=subject_id, values="Mean_Tonic"
)
cross_corr = (
    pivot_check.corr().values[np.triu_indices(pivot_check.shape[1], k=1)].mean()
)
print(
    f"Avg cross-subject correlation (Mean_Tonic, pre-detrend): {cross_corr:.3f}"
)

Avg cross-subject correlation (Mean_Tonic, pre-detrend): 0.074


In [ ]:
df = df[df["condition"] != "black"].reset_index(drop=True)
df["condition"] = df["condition"].replace(
    {"neutral": 0, "positive": 1, "negative": 1}
)
subject_id = df["X"].astype(str).apply(lambda x: x.split("_")[2])
y = df["condition"]
groups = subject_id
class_labels = sorted(y.unique().tolist())
print(f"Rows remaining: {len(df)}")
print(y.value_counts())

Rows remaining: 414
condition
1    276
0    138
Name: count, dtype: int64


/tmp/ipykernel_3898/3129886670.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["condition"] = df["condition"].replace(


Detrended the data as biologicall the signal drifts overtime and this was an attempt to account for that

In [ ]:
gms_summary_cols = [
    "Mean_Phasic",
    "Max_Phasic",
    "Min_Phasic",
    "SD_Phasic",
    "Mean_Tonic",
    "Max_Tonic",
    "Min_Tonic",
    "SD_Tonic",
]
raw_stat_cols = [
    c
    for c in [
        "Max_raw",
        "Min_raw",
        "std_raw",
        "kurtosis_raw",
        "skewness_raw",
        "mean_raw",
    ]
    if c in df.columns
]
cols_to_detrend = gms_summary_cols + raw_stat_cols

time_idx = df.groupby(subject_id).cumcount()
df_detrended = df.copy()

for col in cols_to_detrend:
    detrended_vals = np.zeros(len(df))
    for subj in subject_id.unique():
        mask = (subject_id == subj).values
        t = time_idx[mask].values
        v = df.loc[mask, col].values
        if mask.sum() > 2 and np.std(t) > 0:
            slope, intercept = np.polyfit(t, v, 1)
            detrended_vals[mask] = v - (slope * t + intercept)
        else:
            detrended_vals[mask] = v - np.nanmean(v)
    df_detrended[col + "_dt"] = detrended_vals

detrended_cols = [c + "_dt" for c in cols_to_detrend]

pivot_post = df_detrended.assign(t=time_idx).pivot_table(
    index="t", columns=subject_id, values="Mean_Tonic_dt"
)
cross_corr_post = (
    pivot_post.corr().values[np.triu_indices(pivot_post.shape[1], k=1)].mean()
)
print(
    f"Avg cross-subject correlation (Mean_Tonic_dt, post-detrend): {cross_corr_post:.3f}"
)

row_means_3class = df[signal_cols].mean(axis=1)
X_raw_filled = df[signal_cols].apply(
    lambda row: row.fillna(row_means_3class[row.name]), axis=1
)

Avg cross-subject correlation (Mean_Tonic_dt, post-detrend): -0.005


Used GroupKFold to handle more data leakage problem as it was using other peoples time to predict based o the point in the study they were at rather than the individual patients status. Also did another SVD step and used the singular values as a feature and got noticeable improvements. SVD feature engineering has to be done in each fold otherwise it will leak information outside the training set

In [ ]:
gms_only_cols = [
    c for c in detrended_cols if c.replace("_dt", "") in gms_summary_cols
]
print(f"GMS-only feature columns: {gms_only_cols}")


def run_cv(y_target, label, feature_cols, use_svd=True, verbose=True, params=None):
    """Runs GroupKFold CV with optional hyperparameter overrides."""
    all_y_true, all_y_pred, fold_acc, fold_bal_acc = [], [], [], []
    gkf = GroupKFold(n_splits=5)
    last_model, last_X_train = None, None

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X_raw_filled, y_target, groups)
    ):
        y_train, y_test = y_target.iloc[train_idx], y_target.iloc[test_idx]

        X_train = df_detrended.loc[
            X_raw_filled.iloc[train_idx].index, feature_cols
        ].copy()
        X_test = df_detrended.loc[
            X_raw_filled.iloc[test_idx].index, feature_cols
        ].copy()

        if use_svd:
            svd = TruncatedSVD(n_components=5, random_state=RANDOM_STATE)
            X_train_svd = svd.fit_transform(X_raw_filled.iloc[train_idx])
            X_test_svd = svd.transform(X_raw_filled.iloc[test_idx])
            svd_cols = [f"SVD_{i+1}" for i in range(5)]
            X_train_svd_df = pd.DataFrame(
                X_train_svd,
                columns=svd_cols,
                index=X_raw_filled.iloc[train_idx].index,
            )
            X_test_svd_df = pd.DataFrame(
                X_test_svd,
                columns=svd_cols,
                index=X_raw_filled.iloc[test_idx].index,
            )
            X_train = pd.concat([X_train, X_train_svd_df], axis=1)
            X_test = pd.concat([X_test, X_test_svd_df], axis=1)

        # Baseline parameters updated dynamically
        model_params = {
            "learning_rate": 0.09,
            "max_depth": -1,  # Switched to standard -1 for unrestricted depth
            "random_state": RANDOM_STATE,
            "verbose": -1,
        }
        if params is not None:
            model_params.update(params)

        model = lgb.LGBMClassifier(**model_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        fold_acc.append(accuracy_score(y_test, y_pred))
        fold_bal_acc.append(balanced_accuracy_score(y_test, y_pred))
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)
        last_model, last_X_train = model, X_train

    if verbose:
        print(f"[{label}] Fold accuracies: {[f'{a:.3f}' for a in fold_acc]}")
        print(
            f"[{label}] Mean Accuracy: {np.mean(fold_acc):.3f} | "
            f"Mean Balanced Accuracy: {np.mean(fold_bal_acc):.3f}"
        )
    return all_y_true, all_y_pred, last_model, last_X_train, np.mean(fold_acc)

GMS-only feature columns: ['Mean_Phasic_dt', 'Max_Phasic_dt', 'Min_Phasic_dt', 'SD_Phasic_dt', 'Mean_Tonic_dt', 'Max_Tonic_dt', 'Min_Tonic_dt', 'SD_Tonic_dt']


Optuna is a hyperparameter optimizer

In [ ]:
def objective(trial):
    # Suggest search spaces for tuning LightGBM
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 15, 128),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
    }

    # Evaluate using the full 5-fold CV to ensure robustness without data leakage
    _, _, _, _, mean_bal_accuracy = run_cv(
        y_target=y,
        label="Optuna Trial",
        feature_cols=detrended_cols,
        use_svd=True,
        verbose=False,
        params=params,
    )

    return mean_bal_accuracy


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # Increase trials if you have time/compute

print("\nBest Parameters Found by Optuna:")
print(study.best_params)

[I 2026-07-03 12:50:34,044] A new study created in memory with name: no-name-c0a55e9c-4181-421e-be46-36e80d961327
[I 2026-07-03 12:50:34,356] Trial 0 finished with value: 0.6344444444444444 and parameters: {'learning_rate': 0.05298000468052586, 'max_depth': 7, 'num_leaves': 24, 'min_child_samples': 53, 'subsample': 0.6480151984599133, 'colsample_bytree': 0.7636243555476749}. Best is trial 0 with value: 0.6344444444444444.
[I 2026-07-03 12:50:34,553] Trial 1 finished with value: 0.655 and parameters: {'learning_rate': 0.09168086417635893, 'max_depth': 8, 'num_leaves': 128, 'min_child_samples': 93, 'subsample': 0.7281553567400156, 'colsample_bytree': 0.9130937801368917}. Best is trial 1 with value: 0.655.
[I 2026-07-03 12:50:34,721] Trial 2 finished with value: 0.616111111111111 and parameters: {'learning_rate': 0.017724899301829238, 'max_depth': 4, 'num_leaves': 77, 'min_child_samples': 92, 'subsample': 0.8805114958751266, 'colsample_bytree': 0.9251807976177104}. Best is trial 1 with va


Best Parameters Found by Optuna:
{'learning_rate': 0.05873506390286702, 'max_depth': 8, 'num_leaves': 63, 'min_child_samples': 72, 'subsample': 0.9633913542263736, 'colsample_bytree': 0.8559577138753396}


In [ ]:
# --- Optimized GMS-ONLY run ---
print("\n--- Optimized GMS-only (no SVD raw-signal features) ---")
gms_y_true, gms_y_pred, gms_model, gms_X_train, gms_only_accuracy = run_cv(
    y,
    "GMS-ONLY",
    feature_cols=gms_only_cols,
    use_svd=False,
    params=study.best_params,
)

# --- Optimized Combined run ---
print("\n--- Optimized Combined: GMS + raw stats + SVD ---")
all_y_true, all_y_pred, model, X_train, real_accuracy = run_cv(
    y,
    "REAL (GMS+raw+SVD)",
    feature_cols=detrended_cols,
    use_svd=True,
    params=study.best_params,
)

print(
    f"\nSummary: GMS-only accuracy = {gms_only_accuracy:.3f} | "
    f"Combined (GMS+raw+SVD) accuracy = {real_accuracy:.3f} | "
    f"Chance = {1/len(class_labels):.3f}"
)

# Visualizing Final Results
cm = confusion_matrix(all_y_true, all_y_pred, labels=class_labels)

print("\nClassification Summary Report:")
print(
    classification_report(
        all_y_true,
        all_y_pred,
        target_names=[str(c) for c in class_labels],
    )
)


--- Optimized GMS-only (no SVD raw-signal features) ---
[GMS-ONLY] Fold accuracies: ['0.700', '0.622', '0.578', '0.611', '0.694']
[GMS-ONLY] Mean Accuracy: 0.641 | Mean Balanced Accuracy: 0.504

--- Optimized Combined: GMS + raw stats + SVD ---
[REAL (GMS+raw+SVD)] Fold accuracies: ['0.811', '0.744', '0.700', '0.569', '0.569']
[REAL (GMS+raw+SVD)] Mean Accuracy: 0.679 | Mean Balanced Accuracy: 0.585

Summary: GMS-only accuracy = 0.641 | Combined (GMS+raw+SVD) accuracy = 0.679 | Chance = 0.500

Classification Summary Report:
              precision    recall  f1-score   support

           0       0.56      0.30      0.39       138
           1       0.72      0.88      0.79       276

    accuracy                           0.69       414
   macro avg       0.64      0.59      0.59       414
weighted avg       0.66      0.69      0.66       414

